# BICAM Congressional Hearings — Data Exploration

Key questions from the setup guide:
- What does the transcript text field look like?
- Are speaker names embedded in the transcript?
- How does the format compare to the original paper's source files?
- Any data quality issues?

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="python-dotenv")

# Navigate to project root so 'src' is importable
project_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if not os.path.isdir(os.path.join(project_root, "src")):
    # Fallback: assume cwd is project root or notebooks/
    if os.path.isdir("src"):
        project_root = os.getcwd()
    else:
        project_root = os.path.dirname(os.getcwd())
os.chdir(project_root)
sys.path.insert(0, project_root)
print(f"Project root: {project_root}")

import pandas as pd

from src.data import TEXT_COLUMN, filter_hearings_by_congress, load_hearings, load_hearings_texts_chunked
from src.preprocess import create_sentence_records, download_nltk_deps, segment_speakers

download_nltk_deps()
print("Ready.")

## 1. Hearings Metadata

In [ ]:
hearings = load_hearings()
print(f"Total hearings: {len(hearings)}")
print(f"Columns: {list(hearings.columns)}")
print(f"Congress range: {hearings['congress'].min()} - {hearings['congress'].max()}")
print("\nHearings per congress:")
hearings["congress"].value_counts().sort_index()

In [ ]:
# Filter to our target: 115th+ Congress, House only
new_era = filter_hearings_by_congress(hearings, min_congress=115, chamber="house")
print(f"115th+ House hearings: {len(new_era)}")
print("\nBreakdown by congress:")
new_era["congress"].value_counts().sort_index()

## 2. Transcript Format Inspection

Load a few transcripts and inspect the raw text format.

In [ ]:
target_ids = new_era[new_era["congress"] == 115]["hearing_id"].head(5).tolist()
texts = load_hearings_texts_chunked(target_ids)
print(f"Loaded {len(texts)} transcripts")
print(f"Columns: {list(texts.columns)}")
texts.head()

In [ ]:
# Show first 2000 chars of a sample transcript
row = texts[texts[TEXT_COLUMN].str.len() > 100].iloc[0]
text = row[TEXT_COLUMN]
hearing_id = row["hearing_id"]

print(f"Hearing: {hearing_id}")
print(f"Text length: {len(text):,} chars")
print("=" * 60)
print(text[:2000])

## 3. Speaker Segmentation

Test splitting the transcript into per-speaker chunks.

In [ ]:
chunks = segment_speakers(text)
print(f"Speaker chunks found: {len(chunks)}")
print(f"Unique speakers: {len(set(c['speaker'] for c in chunks))}")
print()
for i, c in enumerate(chunks[:8]):
    print(f"[{i}] {c['speaker']}: {c['text'][:100]}...")

## 4. Sentence Splitting with Context Windows

Split speaker chunks into individual sentences with before/after context.

In [ ]:
records = create_sentence_records(chunks[:3], hearing_id)
print(f"Sentences from first 3 speaker chunks: {len(records)}")
print()
for r in records[:5]:
    print(f"Speaker:  {r['speaker']} (last_name: {r['speaker_last_name']})")
    print(f"Before:   {r['context_before'][:80]}")
    print(f"Target:   {r['target_sentence'][:80]}")
    print(f"After:    {r['context_after'][:80]}")
    print()

## 5. Pipeline Output

Inspect the pre-built pipeline outputs (if available).

In [ ]:
sample_path = os.path.join(os.getcwd(), "data", "sample_for_labeling.csv")
if os.path.exists(sample_path):
    sample = pd.read_csv(sample_path)
    print(f"Silver labeling sample: {len(sample):,} sentences")
    print("\nCongress distribution:")
    print(sample["congress"].value_counts().sort_index())
    print("\nParty distribution:")
    print(sample["party"].value_counts())
    print("\nMinority status (1=minority, 0=majority):")
    print(sample["minority"].value_counts())
    print("\nSample rows:")
    display(
        sample[["hearing_id", "speaker", "target_sentence", "congress", "party", "minority", "committee_name"]].head(10)
    )
else:
    print("No pipeline output found. Run: .venv/bin/python -m src.pipeline")

## 6. Enriched Covariates

Explore the new covariates added during Step 4 of the pipeline (Voteview, Unified Govt, Leadership).

In [ ]:
enriched_path = os.path.join(os.getcwd(), "data", "sentences_enriched.csv")
if os.path.exists(enriched_path):
    enriched = pd.read_csv(enriched_path, low_memory=False)
    print(f"Enriched dataset: {len(enriched):,} sentences")
    
    print("\nGender distribution (1=female):")
    if "female" in enriched.columns:
        print(enriched["female"].value_counts(dropna=False))
        
    print("\nUnified government distribution:")
    if "unified" in enriched.columns:
        print(enriched["unified"].value_counts(dropna=False))
        
    print("\nLeadership status distribution:")
    for col in ["chairspeech", "rankmemspeech", "leader"]:
        if col in enriched.columns:
            print(f"{col}:")
            print(enriched[col].value_counts(dropna=False))
            
    print("\nSample of enriched columns:")
    cols_to_show = ["speaker", "party", "female", "seniority", "nominate_dim1", "unified", "chairspeech", "rankmemspeech", "leader"]
    existing_cols = [c for c in cols_to_show if c in enriched.columns]
    display(enriched[existing_cols].head(10))
else:
    print("No enriched output found. Ensure you have run Step 4 of the pipeline.")